In [1]:
import logging

import httpx

from seis_lab_data.authentik import get_user_by_uuid
from seis_lab_data.config import get_settings
from seis_lab_data.db import queries
from seis_lab_data.operations import discovery as discovery_ops

logging.basicConfig(level=logging.DEBUG)
logging.getLogger("httpcore").setLevel(logging.ERROR)
logging.getLogger("httpx").setLevel(logging.ERROR)

web_client = httpx.AsyncClient()
settings = get_settings()

In [2]:
async with settings.get_db_session_maker()() as session:
    p1 = await queries.get_project_by_english_name(session, "PRR windfarms")
    assert p1 is not None

In [3]:
discovery_owner = await get_user_by_uuid(
    admin_token=settings.auth_admin_token,
    user_id=p1.owner_id,
    web_client=web_client,
    authentik_base_url=settings.auth_internal_base_url,
)
assert discovery_owner is not None

In [4]:
async with settings.get_db_session_maker()() as session:
    new_missions = await discovery_ops.discover_project_contents(
        session=session,
        project=p1,
        settings=settings,
        user=discovery_owner,
    )

DEBUG:seis_lab_data.operations.discovery:mission_path=Path('/data/simulated-archive/surveys/owf-seism-2024')
DEBUG:seis_lab_data.events.emitters:no-op emit event called with event=SeisLabDataEvent(type_=<EventType.SURVEY_MISSION_CREATED: 'survey_mission_created'>, initiator='94291116-7fb7-4889-9d83-8b04718d0248', payload=EventPayload(before=None, after={'id': '1ff04760-8ebd-427a-8a48-b77343d31ef1', 'name': {'en': 'seism-2024', 'pt': None}, 'description': {'en': 'Some description about the seism-2024 survey', 'pt': None}, 'status': <SurveyMissionStatus.DRAFT: 'draft'>, 'validation_result': None, 'project': {'id': '2b663029-2651-4c2f-b4a9-2f53d5d00c41', 'name': {'en': 'PRR windfarms', 'pt': 'PRR Eólicas'}, 'status': <ProjectStatus.DRAFT: 'draft'>, 'validation_result': None, 'root_path': 'surveys', 'bbox_4326': None, 'temporal_extent_begin': '', 'temporal_extent_end': '', 'num_survey_missions': 0, 'num_survey_related_records': 0}, 'temporal_extent_begin': '', 'temporal_extent_end': '', 'b